In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime as dt

In [10]:
default_curr = {'USD':3.65, 'EUR': 4.2}
# Binance Transactions
df = pd.read_csv('bin_trans.csv', delimiter=';')
# extendable by user input
operations_dep = ['Deposit','Buy Crypto With Fiat']
df_input = df[df.Operation.isin(operations_dep)]

# Data wrangling
df_input['Coin'] = df_input['Coin'].apply(lambda x: x[:3])
df_input['Change'] = round(df_input['Change'].astype(float),2)
df_input.rename(columns={'Change':'Value'}, inplace=True)
df_input.reset_index(drop=True, inplace=True)
df_input['UTC_Time'] = df_input['UTC_Time'].str.split(' ').str[0]
df_input['UTC_Time'] = pd.to_datetime(df_input['UTC_Time'])
df_input = df_input.iloc[:,1:-1]
df_input['FX_Change'] = 1
for ind in range(0,len(df_input)):
    curr_ = df_input.iloc[ind,3] 
    if curr_ != "PLN":
        try:
            df_input['FX_Change'][ind] = round(yf.download(f'{curr_}PLN=X',start=df_input.iloc[ind,0], end=df_input.iloc[ind,0] + pd.DateOffset(days=1)).iloc[0,0],2)
        except:
            df_input['FX_Change'][ind] = default_curr['EUR']
df_input['Value_PLN'] = round(df_input.Value * df_input.FX_Change,2)

# End results
df['Change'] = df['Change'].astype(str).str.strip(' ').str.replace(',','.').astype(float)
df_holdings = pd.DataFrame(df[(df['Operation'].isin(['Asset Recovery','Deposit','Buy','Fee','Sell','Stacking Rewards','Transaction Fee','Transaction Sold','Simple Earn Flexible Interest','Transaction Revenue','Binance Convert','Simple Earn Locked Rewards','Transaction Spend','Transfer Between Main and Funding Wallet']))].groupby(by='Coin')['Change'].sum())


df_trans = df[~df['Remark'].isin(["Binance Earn", "Binance Launchpool"])]
df_trans['Change'] = df_trans['Change'].astype(float)
df_trans = pd.merge(df_trans[(df_trans['Operation'] == "Binance Convert") & (df_trans["Change"]>0)].loc[:,['UTC_Time','Operation','Coin','Change']],
         df_trans[(df_trans['Operation'] == "Binance Convert") & (df_trans["Change"]<0)].loc[:,['UTC_Time','Operation','Coin','Change']],
         on = 'UTC_Time').drop_duplicates(keep='first')

/var/folders/x8/q__bzqys7yg57g9twpxbqxqm0000gn/T/ipykernel_1165/1612957293.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_input['Coin'] = df_input['Coin'].apply(lambda x: x[:3])
/var/folders/x8/q__bzqys7yg57g9twpxbqxqm0000gn/T/ipykernel_1165/1612957293.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_input['Change'] = round(df_input['Change'].astype(float),2)
/var/folders/x8/q__bzqys7yg57g9twpxbqxqm0000gn/T/ipykernel_1165/1612957293.py:11: SettingWithCopyWarning: 
A value is trying to be set 

/var/folders/x8/q__bzqys7yg57g9twpxbqxqm0000gn/T/ipykernel_1165/1612957293.py:21: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_input['FX_Change'][ind] = round(yf.download(f'{curr_}PLN=X',start=df_input.iloc[ind,0], end=df_input.iloc[ind,0] + pd.DateOffset(days=1)).iloc[0,0],2)
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['EURPLN=X']: DNSError('Failed to perform, curl: (6) Could not resolve host: query1.finance.yahoo.com. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')
/var/folders/x8/q__bzqys7yg57g9twpxbqxqm0000gn/T/ipykernel_1165/1612957293.py:21: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_input['FX_Change'][ind] = round(yf.download(f'{curr_}PLN=X',start=df_input.iloc[ind,0], end=df_input.iloc[ind,0] + pd.DateOffset(days=1)).iloc[0,0],2)
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['USDPLN=X'

In [11]:
df_trans

,UTC_Time,Operation_x,Coin_x,Change_x,Operation_y,Coin_y,Change_y
0,04/01/2024 17:15,Binance Convert,FDUSD,0.136985,Binance Convert,BUSD,-0.136985
1,20/02/2024 18:21,Binance Convert,BTC,0.005254,Binance Convert,EUR,-250.000000
2,20/02/2024 18:22,Binance Convert,ETH,0.148012,Binance Convert,EUR,-400.000000
3,20/02/2024 18:25,Binance Convert,EUR,11.973713,Binance Convert,ALICE,-10.118320
4,20/02/2024 18:26,Binance Convert,BNB,0.616069,Binance Convert,EUR,-200.000000
...,...,...,...,...,...,...,...
210,01/06/2025 07:20,Binance Convert,BNB,0.000490,Binance Convert,BMT,-3.686701
211,04/07/2025 08:38,Binance Convert,USDC,410.986538,Binance Convert,PLN,-1485.000000
212,25/07/2025 04:15,Binance Convert,BNB,0.548056,Binance Convert,USDC,-414.308696
213,28/07/2025 19:33,Binance Convert,USDC,938.225949,Binance Convert,PLN,-3465.000000


In [130]:
df_maybe = df_test[(~df_test['Change_y'].isna()) | (df_test['Operation_x']=='Deposit')].reset_index(drop=True)

In [210]:
holdings = {}

In [211]:
for i in range(0,len(df_test_2)):
    # add new one
    if df_test_2.iloc[i,2] in holdings:
        new_ = holdings[df_test_2.iloc[i,2]] + df_test_2.iloc[i,3]
        holdings[df_test_2.iloc[i,2]] = new_
    # add to exsisting
    else:
        holdings[df_test_2.iloc[i,2]] = df_test_2.iloc[i,3]
    
    # substract from exsisting
    if df_test_2['Operation_x'][i] != 'Deposit' and pd.notna(df_test_2['Change_y'][i]):
        new_ = holdings[df_test_2.iloc[i,5]] + df_test_2.iloc[i,6]
        holdings[df_test_2.iloc[i,5]] = new_

In [212]:
df_test_2

{'EUR': np.float64(-189.8845600000001),
 'BTC': np.float64(0.035718719999999995),
 'ETH': np.float64(0.32128833000000034),
 'DOGE': np.float64(0.998899999999909),
 'FTM': np.float64(0.09999999999999432),
 'ADA': np.float64(477.5922520899974),
 'USDT': np.float64(-36.02374669999983),
 'FIO': np.float64(-0.1494260000000156),
 'BUSD': np.float64(334.18117600000016),
 'SUSHI': np.float64(-5.501735),
 'AAVE': np.float64(0.11735699999999998),
 'RUNE': np.float64(8.2867),
 'SNX': np.float64(-4.163745),
 'HEGIC': np.float64(184.31),
 'FET': np.float64(169.72820100999945),
 'OCEAN': np.float64(42.28665809),
 'CHZ': np.float64(-0.3806620999999777),
 'MATIC': np.float64(409.02167197999734),
 'ENJ': np.float64(-0.06919999999999857),
 'COTI': np.float64(151.44271053999995),
 'BNB': np.float64(9.543210210000158),
 'ALICE': np.float64(10.07883543),
 'LUNA': np.float64(-8.881784197001252e-16),
 'IOST': np.float64(791.9810559599969),
 'BAR': np.float64(0.00321889),
 'TLM': np.float64(0.6174273999999969

In [217]:
row = pd.DataFrame({'Country':['Poland','Poland','Poland'],
                'Purchase Date':['2025-07-22','2025-07-22','2025-07-22'],
                'Quantity':[20,20,5],
                'Interest Rate':[0.05,0.05,0.0475],
                "Purchase Price":[100,100,100]})

In [182]:
df[df['Coin']=='LUNA']

,User_ID,UTC_Time,Account,Operation,Coin,Change,Remark
99,71732789,09/04/2021 08:29,Spot,Buy,LUNA,3.80,NaN
134,71732789,21/04/2021 16:09,Spot,Buy,LUNA,8.10,NaN
208,71732789,30/10/2021 15:01,Spot,Transaction Sold,LUNA,-9.23,NaN
240,71732789,22/01/2022 10:02,Spot,Transaction Buy,LUNA,2.07,NaN
247,71732789,27/01/2022 21:54,Spot,Transaction Buy,LUNA,2.15,NaN
259,71732789,05/02/2022 02:59,Spot,Transaction Sold,LUNA,-6.89,NaN


In [204]:
df_test_2[(df_test_2['Coin_x']=='LUNA') | (df_test_2['Coin_y']=='LUNA')]

,UTC_Time,Operation_x,Coin_x,Change_x,Operation_y,Coin_y,Change_y
75,09/04/2021 08:29,Buy,LUNA,3.800,Sell,BUSD,-63.304200
101,21/04/2021 16:09,Buy,LUNA,8.100,Sell,BNB,-0.196830
102,21/04/2021 16:09,Buy,LUNA,8.100,Fee,BNB,-0.000148
168,30/10/2021 15:01,Transaction Revenue,EUR,346.125,Transaction Sold,LUNA,-9.230000
191,22/01/2022 10:02,Transaction Buy,LUNA,2.070,Transaction Spend,BUSD,-113.850000
197,27/01/2022 21:54,Transaction Buy,LUNA,2.150,Transaction Spend,BUSD,-112.875000
207,05/02/2022 02:59,Transaction Revenue,USDT,385.151,Transaction Sold,LUNA,-6.890000


In [209]:
df_test_2 = pd.merge(df[(df['Change'] > 0 ) & (df['Operation']!='Transaction Fee') & (df['Operation']!='Fee')].loc[:,['UTC_Time','Operation','Coin','Change']],
         df[(df['Change'] < 0 ) & (df['Operation']!='Transaction Fee') & (df['Operation']!='Fee')].loc[:,['UTC_Time','Operation','Coin','Change']],
         on = 'UTC_Time', how ='left')

In [199]:
a = df[(df['Change'] < 0 )].loc[:,['UTC_Time','Operation','Coin','Change']]
a[a['Coin']=="LUNA"]

,UTC_Time,Operation,Coin,Change
208,30/10/2021 15:01,Transaction Sold,LUNA,-9.23
259,05/02/2022 02:59,Transaction Sold,LUNA,-6.89


In [176]:
df_test['Operation_x'].unique()

array(['Deposit', 'Buy', 'Launchpool Subscription/Redemption',
       'Launchpool Earnings Withdrawal', 'Staking Rewards',
       'Launchpad Token Distribution', 'Launchpad Subscribe',
       'Staking Redemption', 'Airdrop Assets', 'Transaction Revenue',
       'Transaction Buy', 'Simple Earn Locked Rewards',
       'Simple Earn Flexible Interest', 'BNB Vault Rewards',
       'Simple Earn Locked Redemption', 'Distribution',
       'Simple Earn Flexible Redemption', 'Binance Convert', 'Crypto Box',
       'Launchpool Airdrop', 'Buy Crypto With Fiat', 'Megadrop Rewards',
       'Token Swap - Distribution', 'HODLer Airdrops Distribution',
       'Transfer Between Main and Funding Wallet'], dtype=object)

In [177]:
df_test[df_test['Operation_x'].isin(['Launchpool Subscription/Redemption','Launchpool Earnings Withdrawal','Simple Earn Locked Redemption','Simple Earn Flexible Redemption'])]

,UTC_Time,Operation_x,Coin_x,Change_x,Operation_y,Coin_y,Change_y
37,09/04/2021 00:43,Launchpool Subscription/Redemption,BUSD,63.316800,NaN,NaN,NaN
38,09/04/2021 00:58,Launchpool Earnings Withdrawal,ALICE,0.078835,NaN,NaN,NaN
95,07/05/2021 00:21,Launchpool Subscription/Redemption,BNB,0.293700,NaN,NaN,NaN
96,07/05/2021 00:22,Launchpool Earnings Withdrawal,TLM,2.797427,NaN,NaN,NaN
1104,23/08/2023 01:08,Simple Earn Locked Redemption,FIO,0.576380,NaN,NaN,NaN
1417,23/11/2023 19:39,Simple Earn Flexible Redemption,ADA,155.494048,NaN,NaN,NaN
1418,23/11/2023 19:39,Simple Earn Flexible Redemption,ALICE,10.118320,NaN,NaN,NaN
1419,23/11/2023 19:40,Simple Earn Flexible Redemption,BTC,0.020635,NaN,NaN,NaN
1420,23/11/2023 19:40,Simple Earn Flexible Redemption,BNB,0.071982,NaN,NaN,NaN
1421,23/11/2023 19:40,Simple Earn Flexible Redemption,CHZ,196.990662,NaN,NaN,NaN


In [144]:
holdings

{'EUR': np.float64(5125.96650038),
 'PLN': np.float64(6435.0),
 'BTC': np.float64(0.053076299999999986),
 'ETH': np.float64(0.48051398),
 'DOGE': np.float64(2132.5663),
 'FTM': np.float64(195.94400000000002),
 'ADA': np.float64(419.82485696000003),
 'USDT': np.float64(5250.0913341899995),
 'FIO': np.float64(695.08362),
 'BUSD': np.float64(1574.9906376000001),
 'SUSHI': np.float64(2.755),
 'AAVE': np.float64(0.23723799999999998),
 'RUNE': np.float64(16.619999999999997),
 'SNX': np.float64(2.085),
 'HEGIC': np.float64(184.31),
 'FET': np.float64(171.7),
 'OCEAN': np.float64(82.64160000000001),
 'CHZ': np.float64(196.61),
 'MATIC': np.float64(493.4171584),
 'ENJ': np.float64(43.6),
 'COTI': np.float64(152.25209999999998),
 'BNB': np.float64(2.55948666),
 'LUNA': np.float64(16.119999999999997),
 'IOST': np.float64(824.0),
 'SXP': np.float64(560.89283654),
 'TLM': np.float64(335.82),
 'SOL': np.float64(0.83),
 'ALICE': np.float64(20.118319659999997),
 'TRY': np.float64(183.8382),
 'FDUSD': 

In [27]:
df_test = pd.merge(df[(df['Change'] > 0 )].loc[:,['UTC_Time','Operation','Coin','Change']],
         df[(df['Change'] < 0 )].loc[:,['UTC_Time','Operation','Coin','Change']],
         on = 'UTC_Time', how ='left').drop_duplicates(keep='first')

In [28]:
df_test = df_test.drop_duplicates(
    subset=['UTC_Time', 'Operation_x', 'Coin_x', 'Change_x'], 
    keep='first'
).reset_index(drop=True)

In [57]:
df_coins = pd.DataFrame(df[(df['Operation'].isin(['Asset Recovery','Deposit','Buy','Fee','Sell','Stacking Rewards','Transaction Fee','Transaction Sold','Simple Earn Flexible Interest','Transaction Revenue','Binance Convert','Simple Earn Locked Rewards','Transaction Spend','Transfer Between Main and Funding Wallet']))].groupby(by='Coin')['Change'].sum())

In [61]:
df_coins_2 = pd.DataFrame(df[~(df['Operation'].isin(['Asset Recovery','Deposit','Buy','Fee','Sell','Stacking Rewards','Transaction Fee','Transaction Sold','Simple Earn Flexible Interest','Transaction Revenue','Binance Convert','Simple Earn Locked Rewards','Transaction Spend','Transfer Between Main and Funding Wallet']))])#.groupby(by='Coin')['Change'].sum())

In [66]:
df_coins_2[df_coins_2['Remark'] != 'NaN'].head(45)

,User_ID,UTC_Time,Account,Operation,Coin,Change,Remark
52,71732789,09/03/2021 12:41,Spot,Launchpool Subscription/Redemption,BUSD,-63.316800,Binance Launchpool
91,71732789,06/04/2021 22:45,Spot,Launchpool Subscription/Redemption,BNB,-0.148800,Binance Launchpool
95,71732789,07/04/2021 06:38,Spot,Launchpool Subscription/Redemption,BNB,-0.144900,Binance Launchpool
96,71732789,09/04/2021 00:43,Spot,Launchpool Subscription/Redemption,BUSD,63.316800,Binance Launchpool
97,71732789,09/04/2021 00:58,Spot,Launchpool Earnings Withdrawal,ALICE,0.078835,Binance Launchpool
106,71732789,15/04/2021 09:30,Spot,Staking Purchase,MATIC,-148.251600,NaN
107,71732789,15/04/2021 09:33,Spot,Staking Purchase,IOST,-823.176000,NaN
108,71732789,15/04/2021 16:09,Spot,Staking Purchase,ADA,-53.900000,NaN
109,71732789,17/04/2021 02:51,Spot,Staking Rewards,IOST,0.619975,NaN
110,71732789,17/04/2021 08:26,Spot,Staking Rewards,MATIC,0.127903,NaN


In [29]:
holdings = {}
holdings_in = {}
holdings_out = {}
for line in range(0,20):

    if df_test['Coin_x'][line] not in holdings:
        holdings.update({df_test['Coin_x'][line]:df_test['Change_x'][line]})
    else:
        holdings[df_test['Coin_x'][line]] = holdings[df_test['Coin_x'][line]] + df_test['Change_x'][line]
        
    if str(df_test['Coin_y'][line]) != 'nan':
        holdings[df_test['Coin_y'][line]] = holdings[df_test['Coin_y'][line]] + df_test['Change_y'][line]

    if df_test['Coin_x'][line] not in holdings_in:
        holdings_in.update({df_test['Coin_x'][line]:df_test['Change_x'][line]})
        holdings_out.update({df_test['Coin_x'][line]:0})
    else:
        holdings_in[df_test['Coin_x'][line]] = holdings_in[df_test['Coin_x'][line]] + df_test['Change_x'][line]

    if str(df_test['Coin_y'][line]) != 'nan':
        holdings_out[df_test['Coin_y'][line]] = holdings_out[df_test['Coin_y'][line]] + df_test['Change_y'][line]


In [158]:
df_lol = df[~df['Remark'].isin(["Binance Earn", "Binance Launchpool"])]
df_lol['Change'] = df_lol['Change'].astype(float)
df_lol.reset_index(drop=True, inplace=True)

df_lol['Holdings'] = 1

/var/folders/x8/q__bzqys7yg57g9twpxbqxqm0000gn/T/ipykernel_1165/1629930244.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_lol['Change'] = df_lol['Change'].astype(float)
/var/folders/x8/q__bzqys7yg57g9twpxbqxqm0000gn/T/ipykernel_1165/1629930244.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_lol['Holdings'] = 1


In [159]:
df_lol['Holdings'] = 1
for row_ in range(0,len(df_lol)):
    if row_ == 0:
        df_lol['Holdings'][row_] = {df_lol['Coin'][row_]:df_lol['Change'][row_]}
    else:
        df_lol['Holdings'][row_] = df_lol['Holdings'][row_-1].copy()
        
        if df_lol['Coin'][row_] in df_lol['Holdings'][row_]:
            df_lol['Holdings'][row_][df_lol['Coin'][row_]] = float(df_lol['Holdings'][row_][df_lol['Coin'][row_]]) + float(df_lol['Change'][row_])
        else:
            df_lol['Holdings'][row_][df_lol['Coin'][row_]] = float(df_lol['Change'][row_])

/var/folders/x8/q__bzqys7yg57g9twpxbqxqm0000gn/T/ipykernel_1165/2755235103.py:3: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_lol['Holdings'][row_] = {df_lol['Coin'][row_]:df_lol['Change'][row_]}
/var/folders/x8/q__bzqys7yg57g9twpxbqxqm0

In [203]:
df = pd.read_csv('bin_trans.csv', delimiter=';')
df

,User_ID,UTC_Time,Account,Operation,Coin,Change,Remark
0,71732789,17/02/2021 11:48,Spot,Deposit,EUR,240.00000000,NaN
1,71732789,18/02/2021 11:05,Spot,Sell,EUR,-49.89270000,NaN
2,71732789,18/02/2021 11:05,Spot,Buy,BTC,0.00116300,NaN
3,71732789,18/02/2021 11:06,Spot,Sell,EUR,-49.99009500,NaN
4,71732789,18/02/2021 11:06,Spot,Buy,ETH,0.03147000,NaN
...,...,...,...,...,...,...,...
4467,71732789,11/09/2025 04:44,Spot,HODLer Airdrops Distribution,HOLO,5.00173953,Binance Launchpool
4468,71732789,12/09/2025 04:14,Spot,Simple Earn Flexible Interest,BTC,"0,00",Binance Earn
4469,71732789,12/09/2025 04:20,Spot,Simple Earn Flexible Interest,ETH,4.0E-7,Binance Earn
4470,71732789,12/09/2025 04:22,Spot,Simple Earn Flexible Interest,USDC,0.14848717,Binance Earn


In [223]:
df['Operation'].value_counts()

Operation
Simple Earn Flexible Interest               2165
Simple Earn Locked Rewards                  1444
BNB Vault Rewards                            187
Binance Convert                              176
Staking Rewards                               90
Sell                                          44
Buy                                           44
HODLer Airdrops Distribution                  38
Fee                                           31
Launchpool Subscription/Redemption            31
Launchpool Airdrop                            28
Transaction Revenue                           28
Transaction Sold                              28
Transaction Fee                               24
Simple Earn Flexible Subscription             20
Simple Earn Flexible Redemption               17
Simple Earn Locked Subscription               11
Transaction Buy                                9
Transaction Spend                              9
Deposit                                        6
Asset Reco

In [187]:
df = pd.read_csv('bin_trans.csv', delimiter=';')

df['Holdings'] = 1
df['Change'] = df['Change'].str.replace(',','.').str.strip()

for row_ in range(0,len(df)):
    if row_ == 0:
        df['Holdings'][row_] = {df['Coin'][row_]:df['Change'][row_]}
    else:
        df['Holdings'][row_] = df['Holdings'][row_-1].copy()
        
        if df['Coin'][row_] in df['Holdings'][row_]:
            df['Holdings'][row_][df['Coin'][row_]] = float(df['Holdings'][row_][df['Coin'][row_]]) + float(df['Change'][row_])
        else:
            df['Holdings'][row_][df['Coin'][row_]] = float(df['Change'][row_])

dfff = df[['UTC_Time','Holdings']]
dfff['No_of_coins_hold'] = dfff['Holdings'].apply(lambda x: len(x))
dfff = pd.json_normalize(dfff['Holdings']).fillna(0).round(6)
dfff

/var/folders/x8/q__bzqys7yg57g9twpxbqxqm0000gn/T/ipykernel_1165/3899370233.py:8: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df['Holdings'][row_] = {df['Coin'][row_]:df['Change'][row_]}
/var/folders/x8/q__bzqys7yg57g9twpxbqxqm0000gn/T/ipyk

,EUR,BTC,ETH,DOGE,FTM,ADA,USDT,FIO,BUSD,SUSHI,...,TREE,TOWNS,PROVE,PLUME,DOLO,MITO,SOMI,OPEN,LINEA,HOLO
0,240.00000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
1,190.1073,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
2,190.1073,0.001163,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
3,140.117205,0.001163,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
4,140.117205,0.001163,0.031470,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4467,-0.0,-0.000020,0.000838,-0.0,-0.0,3.289515,-0.0,0.0,-0.0,0.0,...,1.581207,38.665339,1.829287,22.501831,2.408542,2.408552,4.806129,1.616854,118.051345,5.00174
4468,-0.0,-0.000020,0.000838,-0.0,-0.0,3.289515,-0.0,0.0,-0.0,0.0,...,1.581207,38.665339,1.829287,22.501831,2.408542,2.408552,4.806129,1.616854,118.051345,5.00174
4469,-0.0,-0.000020,0.000838,-0.0,-0.0,3.289515,-0.0,0.0,-0.0,0.0,...,1.581207,38.665339,1.829287,22.501831,2.408542,2.408552,4.806129,1.616854,118.051345,5.00174
4470,-0.0,-0.000020,0.000838,-0.0,-0.0,3.289515,-0.0,0.0,-0.0,0.0,...,1.581207,38.665339,1.829287,22.501831,2.408542,2.408552,4.806129,1.616854,118.051345,5.00174


In [226]:
df = pd.read_csv('bin_trans.csv', delimiter=';')

df['Holdings'] = 1
df['Change'] = df['Change'].str.replace(',','.').str.strip()

for row_ in range(0,len(df)):
    if row_ == 0:
        df['Holdings'][row_] = {df['Coin'][row_]:df['Change'][row_]}
    else:
        df['Holdings'][row_] = df['Holdings'][row_-1].copy()
        
        if df['Coin'][row_] in df['Holdings'][row_]:
            df['Holdings'][row_][df['Coin'][row_]] = float(df['Holdings'][row_][df['Coin'][row_]]) + float(df['Change'][row_])
        else:
            df['Holdings'][row_][df['Coin'][row_]] = float(df['Change'][row_])

df['No_of_coins_hold'] = df['Holdings'].apply(lambda x: len(x))
df

/var/folders/x8/q__bzqys7yg57g9twpxbqxqm0000gn/T/ipykernel_1165/1277197050.py:8: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df['Holdings'][row_] = {df['Coin'][row_]:df['Change'][row_]}
/var/folders/x8/q__bzqys7yg57g9twpxbqxqm0000gn/T/ipyk

,User_ID,UTC_Time,Account,Operation,Coin,Change,Remark,Holdings,No_of_coins_hold
0,71732789,17/02/2021 11:48,Spot,Deposit,EUR,240.00000000,NaN,{'EUR': '240.00000000'},1
1,71732789,18/02/2021 11:05,Spot,Sell,EUR,-49.89270000,NaN,{'EUR': 190.1073},1
2,71732789,18/02/2021 11:05,Spot,Buy,BTC,0.00116300,NaN,"{'EUR': 190.1073, 'BTC': 0.001163}",2
3,71732789,18/02/2021 11:06,Spot,Sell,EUR,-49.99009500,NaN,"{'EUR': 140.117205, 'BTC': 0.001163}",2
4,71732789,18/02/2021 11:06,Spot,Buy,ETH,0.03147000,NaN,"{'EUR': 140.117205, 'BTC': 0.001163, 'ETH': 0....",3
...,...,...,...,...,...,...,...,...,...
4467,71732789,11/09/2025 04:44,Spot,HODLer Airdrops Distribution,HOLO,5.00173953,Binance Launchpool,"{'EUR': -1.4210854715202004e-13, 'BTC': -2.027...",110
4468,71732789,12/09/2025 04:14,Spot,Simple Earn Flexible Interest,BTC,0.00,Binance Earn,"{'EUR': -1.4210854715202004e-13, 'BTC': -2.027...",110
4469,71732789,12/09/2025 04:20,Spot,Simple Earn Flexible Interest,ETH,4.0E-7,Binance Earn,"{'EUR': -1.4210854715202004e-13, 'BTC': -2.027...",110
4470,71732789,12/09/2025 04:22,Spot,Simple Earn Flexible Interest,USDC,0.14848717,Binance Earn,"{'EUR': -1.4210854715202004e-13, 'BTC': -2.027...",110


In [235]:
df[df['Remark'].isin(['Ref - N014643682810007070720313','Token swap distribution','Binance Pay'])]

,User_ID,UTC_Time,Account,Operation,Coin,Change,Remark,Holdings,No_of_coins_hold
1633,71732789,25/10/2023 15:14,Spot,Distribution,USDT,0.02117000,Token swap distribution,"{'EUR': 0.000682999999962703, 'BTC': 0.0, 'ETH...",37
1947,71732789,20/02/2024 18:28,Funding,Crypto Box,DEGO,0.00642394,Binance Pay,"{'EUR': 46.20902718999986, 'BTC': 0.00629889, ...",41
2038,71732789,13/03/2024 11:03,Spot,Buy Crypto With Fiat,USDT,861.33363011,Ref - N014643682810007070720313,"{'EUR': 46.20902718999986, 'BTC': -1.095999999...",43
2447,71732789,25/05/2024 16:09,Funding,Crypto Box,FDUSD,0.01200000,Binance Pay,"{'EUR': -1.4210854715202004e-13, 'BTC': -1.499...",50


In [252]:
df[~df['Operation'].isin(['Sell','Buy','Deposit','Fee'])].iloc[75:120,:]

,User_ID,UTC_Time,Account,Operation,Coin,Change,Remark,Holdings,No_of_coins_hold
197,71732789,03/06/2021 03:25,Spot,Staking Rewards,BNB,0.00028032,NaN,"{'EUR': 0.005370999999996684, 'BTC': 7.9999999...",27
198,71732789,04/06/2021 06:21,Spot,Staking Rewards,BNB,0.00028032,NaN,"{'EUR': 0.005370999999996684, 'BTC': 7.9999999...",27
199,71732789,05/06/2021 01:56,Spot,Staking Redemption,BNB,0.29410361,NaN,"{'EUR': 0.005370999999996684, 'BTC': 7.9999999...",27
200,71732789,19/10/2021 03:46,Spot,Airdrop Assets,DON,0.03543848,NaN,"{'EUR': 0.005370999999996684, 'BTC': 7.9999999...",28
201,71732789,30/10/2021 14:42,Spot,Transaction Revenue,BUSD,55.79600000,NaN,"{'EUR': 0.005370999999996684, 'BTC': 7.9999999...",28
202,71732789,30/10/2021 14:42,Spot,Transaction Sold,MATIC,-29.60000000,NaN,"{'EUR': 0.005370999999996684, 'BTC': 7.9999999...",28
203,71732789,30/10/2021 14:52,Spot,Transaction Revenue,BUSD,39.20800000,NaN,"{'EUR': 0.005370999999996684, 'BTC': 7.9999999...",28
204,71732789,30/10/2021 14:52,Spot,Transaction Sold,MATIC,-20.80000000,NaN,"{'EUR': 0.005370999999996684, 'BTC': 7.9999999...",28
205,71732789,30/10/2021 14:52,Spot,Transaction Sold,MATIC,-61.40000000,NaN,"{'EUR': 0.005370999999996684, 'BTC': 7.9999999...",28
206,71732789,30/10/2021 14:52,Spot,Transaction Revenue,BUSD,115.73900000,NaN,"{'EUR': 0.005370999999996684, 'BTC': 7.9999999...",28


In [202]:
dfff.iloc[-1,:][dfff.iloc[-1,:] > 0]

ETH         0.000838
ADA         3.289515
BNB         0.000173
IOST        0.217703
BAR         0.003219
TLM         0.617427
DON         0.035438
ETHW         0.01573
RDNT        0.081283
POL         2.326047
S              0.002
USDC        0.925827
LAYER       1.465527
SHELL       4.033871
RESOLV      2.409596
HOME       24.193749
SPK        24.352012
NEWT        1.524118
SAHARA     15.234018
LA          1.832781
ERA         2.435266
C           2.428965
TREE        1.581207
TOWNS      38.665339
PROVE       1.829287
PLUME      22.501831
DOLO        2.408542
MITO        2.408552
SOMI        4.806129
OPEN        1.616854
LINEA     118.051345
HOLO         5.00174
Name: 4471, dtype: object

In [160]:
df_t = df_lol[['UTC_Time','Holdings']]
df_t['No_of_coins_hold'] = df_t['Holdings'].apply(lambda x: len(x))

/var/folders/x8/q__bzqys7yg57g9twpxbqxqm0000gn/T/ipykernel_1165/953975078.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_t['No_of_coins_hold'] = df_t['Holdings'].apply(lambda x: len(x))


In [166]:
df_holds = pd.json_normalize(df_t['Holdings']).fillna(0).round(6)

In [174]:
import requests

def get_price(symbol):
    url = f"https://api.binance.com/api/v3/ticker/price?symbol={symbol}"
    response = requests.get(url)
    data = response.json()
    return data#['price'])

# Example usage
print(get_price("BTCUSDT"))  # Output: current BTC price in USDT   

{'symbol': 'BTCUSDT', 'price': '85681.34000000'}


In [171]:
df_holds.iloc[-1,:][df_holds.iloc[-1,:] > 0]

BTC        0.003839
ETH        0.144374
ADA      264.507523
BNB        2.706244
BAR        0.003219
SXP      560.892837
DON        0.035438
ETHW       0.015730
RDNT       0.081283
POL      125.196179
S          0.002000
USDC    1341.675507
Name: 739, dtype: float64